In [27]:
!pip install sentence-transformers==2.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.0 MB/s eta 0:0

In [32]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.vectorstores import FAISS

In [29]:
# If needed:
# !pip install -q charset-normalizer langchain-community

from charset_normalizer import from_path
import pandas as pd

csv_path = "/kaggle/input/q-ands-a/codebasics_faqs.csv"

# 1) Detect encoding
enc = (from_path(csv_path).best() or from_path(csv_path).first()).encoding or "latin1"

# 2) Read CSV (handle both new/old pandas)
read_kwargs = {"encoding": enc}
try:
    df = pd.read_csv(csv_path, on_bad_lines="skip", **read_kwargs)
except TypeError:
    # for older pandas
    df = pd.read_csv(csv_path, error_bad_lines=False, **read_kwargs)

# 3) Turn into Documents
try:
    from langchain_community.document_loaders import DataFrameLoader
except ImportError:
    # Fallback if langchain_community isn't available
    from langchain.document_loaders import DataFrameLoader

loader = DataFrameLoader(df, page_content_column="prompt")  # <- use your text column
docs = loader.load()

docs[:2]  # quick peek


[Document(metadata={'response': 'Yes, this is the perfect bootcamp for anyone who has never done coding and wants to build a career in the IT/Data Analytics industry or just wants to perform better in your current job or business using data.'}, page_content='I have never done programming in my life. Can I take this bootcamp?'),
 Document(metadata={'response': 'Till now 9000 + learners have benefitted from the quality of our courses. You can check the review section and also we have attached their LinkedIn profiles so that you can connect with them and ask directly.'}, page_content='Why should I trust Codebasics?')]

In [35]:
instructor_embedding = HuggingFaceInstructEmbeddings()
vectordb = FAISS.from_documents(documents=docs, embedding=instructor_embedding)

/tmp/ipykernel_36/3502497255.py:1: LangChainDeprecationWarning: Default values for HuggingFaceInstructEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceInstructEmbeddings constructor instead.
  instructor_embedding = HuggingFaceInstructEmbeddings()


TypeError: INSTRUCTOR._load_sbert_model() got an unexpected keyword argument 'token'

In [37]:
retriever = vectordb.as_retriever()
redocs = retriever.get_relevant_document("How long is this course valid")


NameError: name 'vectordb' is not defined

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Hello! I'm Qwen, a large-scale language model independently developed by the Tongyi Lab under Alibaba Group. I'm proficient in 100 languages, including Chinese, English, German,
